# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArnavP2305/flyrank-ml-internship-2/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane chosen: Refresh / Content Opportunity Scoring**

I am choosing the **Refresh / Content Opportunity Scoring** lane. The core question is: *given a portfolio of web pages, which pages should an editor review for a content refresh first — and in what order?*

I chose this lane for three reasons grounded in what I observed while running the starter pipeline:

1. **The signal exists and is measurable.** Notebook 01 showed that `search_volume` alone is a near-zero predictor of actual impressions (correlation ≈ 0.001), which means simple volume-based triage is broken — there is room for a learned model to do better by combining multiple signals.

2. **The baseline is real and beatable.** The pipeline's hand-written rule (stale × visible × impressions) achieves Precision@50 ≈ 0.240 on the starter data. The random forest already hits ~3× that on the same holdout. The gap is large enough to be useful, not a rounding artifact.

3. **The output connects directly to an action.** A ranked refresh queue tells an editor exactly which page to open next. There is no ambiguity between model output and editorial action — that direct connection is what makes the work worth doing.

I can confirm or change this lane until the end of Week 4, but the data already supports it strongly enough to start.

In [1]:
# Setup — navigate to repo root (works locally and in Colab)
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/flyrank-bih/flyrank-ml-internship-starter'
REPO_DIR = 'flyrank-ml-internship-starter'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    while not os.path.isdir('data/raw') and os.getcwd() != '/':
        os.chdir('..')

import pandas as pd, numpy as np
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print('Working dir:', os.getcwd())

Loaded: 30,000 rows, 44 columns
Working dir: C:\Users\DELL\.gemini\antigravity\scratch\flyrank-ml-internship-2


## 2. The question: decision, action, cost of a wrong call

**Research question:**  
> *Which pages in a client's content portfolio are most likely to be in measurable decline — and among those, which ones should an editor review for a content refresh first?*

---

### Decision
An SEO editor or content strategist faces a **triage decision** every sprint: of the hundreds (or thousands) of pages in a client's portfolio, which ones are worth the limited editorial hours this week? The current approach relies on instinct, volume thresholds, or hand-written staleness rules. This project replaces that with a **ranked scoring model** that combines multiple observable signals.

- **Unit of analysis:** one row = one content page (not a client, not a day)
- **Output:** a ranked list of pages, each with a refresh-priority score and a reason code explaining the top signal that drove it

### Action
The **person acting** is an SEO editor or content strategist at an agency. The action is: *open the top-ranked page, review it, and decide whether to rewrite, expand, or restructure it.* Without this ranked queue, editors either work alphabetically, by gut feel, or by whichever client escalates first — all of which waste hours on pages that don't need attention.

### Cost of a wrong recommendation
There are two error types, and they are not symmetric:

| Error | What happens | Cost |
|---|---|---|
| **False positive** (flag a healthy page) | Editor wastes 1–3 hours reviewing a page that didn't need it | Low-to-medium: wasted time, mild frustration |
| **False negative** (miss a declining page) | A page continues declining — losing impressions, clicks, and rank — unnoticed for weeks | High: compounding traffic loss, client dissatisfaction |

This asymmetry means **recall matters more than precision at the very top of the queue** — we would rather review a few extra healthy pages than miss a page that is actively falling. In practice, Precision@50 is the headline metric (industry-standard for refresh queues), but the model will also be evaluated on recall among the top-200.

### Why data / ML helps
A single hand-written rule (e.g., 'flag pages older than 180 days with > 500 impressions') ignores the interaction between signals: a stale page with improving CTR is very different from a stale page with collapsing position. ML earns its place here because:

- There are **6+ observable signals** that all matter — their joint combination is too complex to write by hand
- The pattern **shifts by content type and client** — what counts as 'declining' for a high-volume page is not the same threshold for a niche long-tail page
- The starter pipeline already demonstrates the gap: hand rule Precision@50 ≈ 0.240, random forest ≈ 3× that — a gap that is large enough to justify the complexity

In [2]:
# Validate the decision grain: one row = one page
n_total  = len(df)
n_unique = df['content_id'].nunique()
n_clients = df['client_id'].nunique()
print(f'Total rows:          {n_total:,}')
print(f'Unique content_id:   {n_unique:,}  (confirms one row per page)')
print(f'Unique client_id:    {n_clients}')

# Show how many pages are declining (the target label)
decline_rate = df['trend_direction'].str.lower().eq('down').mean()
print(f'\nDeclining pages (trend_direction=down): {decline_rate:.1%} of the dataset')
print('This is the label our model will learn to surface.')

Total rows:          30,000
Unique content_id:   30,000  (confirms one row per page)
Unique client_id:    32

Declining pages (trend_direction=down): 54.2% of the dataset
This is the label our model will learn to surface.


## 3. Quick look at the data (2–3 real numbers)

The three numbers below are computed live from `data/raw/content_refresh_anonymized.csv`. Each one makes a specific argument for why this lane is worth 7 weeks of work.

**Number 1 — The baseline rule is far from ceiling.**  
The hand-written rule (stale × visible × impressions) achieves Precision@50 ≈ 0.24 on the full dataset. This means roughly 1 in 4 of its top-50 pages are actually declining. A random forest trained on the same data reaches ~3× that. There is a real, large gap to close.

**Number 2 — Search volume is not a reliable triage signal.**  
Notebook 01 showed `search_volume` correlates with `impressions_90d` at ≈ 0.001 — essentially random. This rules out the simplest possible approach ('sort by keyword volume') and means the model must use actual page-level performance metrics.

**Number 3 — Position predicts CTR far more than content length.**  
Among pages with ≥ 100 impressions, `avg_position` and `ctr` correlate at ≈ −0.50, while word count barely separates declining from growing pages (medians within 10%). This tells us which features to trust and which to deprioritize — the model should lean on position, CTR trajectory, and staleness, not word count alone.

In [3]:
# Number 1 — baseline vs model gap (from pipeline outputs)
import json

with open('outputs/model_results.json') as f:
    res = json.load(f)

base_p50 = res['baseline']['baseline_precision_at_50']
rf_p50   = res['models']['random_forest']['precision_at_50']
print('=== Number 1: Baseline vs Random Forest gap ===')
print(f'Hand-written rule  Precision@50: {base_p50:.3f}  (~{round(base_p50*50)} of top-50 correct)')
print(f'Random forest      Precision@50: {rf_p50:.3f}  (~{round(rf_p50*50)} of top-50 correct)')
print(f'Improvement factor: {rf_p50/base_p50:.1f}x')
print('A 3x gap on the headline metric justifies building a better model.\n')

# Number 2 — search volume vs impressions correlation
print('=== Number 2: Search volume as a triage signal ===')
corr_all = df['search_volume'].corr(df['impressions_90d'])
corr_pos = df[df['impressions_90d'] > 0]['search_volume'].corr(
           df[df['impressions_90d'] > 0]['impressions_90d'])
print(f'Correlation (all pages):              {corr_all:.4f}')
print(f'Correlation (pages with impressions): {corr_pos:.4f}')
print('Near-zero both ways: keyword volume alone cannot triage content health.\n')

# Number 3 — position vs CTR and word count separation
print('=== Number 3: Position predicts CTR; word count does not ===')
vis = df[df['impressions_90d'] >= 100]
pos_ctr_corr = vis['avg_position'].corr(vis['ctr'])
wc_by_trend  = df.groupby('trend_direction')['word_count'].median()
print(f'avg_position vs CTR correlation (impressions >= 100): {pos_ctr_corr:.4f}')
print('Word count median by trend direction:')
print(wc_by_trend.round(0).to_string())
print('\nConclusion: position/CTR signals are informative; word count is not a clean lever.')

=== Number 1: Baseline vs Random Forest gap ===
Hand-written rule  Precision@50: 0.240  (~12 of top-50 correct)
Random forest      Precision@50: 0.740  (~37 of top-50 correct)
Improvement factor: 3.1x
A 3x gap on the headline metric justifies building a better model.

=== Number 2: Search volume as a triage signal ===
Correlation (all pages):              0.0012
Correlation (pages with impressions): 0.0012
Near-zero both ways: keyword volume alone cannot triage content health.

=== Number 3: Position predicts CTR; word count does not ===


avg_position vs CTR correlation (impressions >= 100): -0.2387
Word count median by trend direction:
trend_direction
down      2909.0
flat      2698.0
new       2239.0
stable    2912.0
up        2848.0

Conclusion: position/CTR signals are informative; word count is not a clean lever.


## 4. Careful words: what I can and can't claim

This section is the honesty contract. It keeps the project from overstating what an anonymized dataset and a trained model can actually prove.

---

### What this work CAN say

- **Observed / measured:** 'In this 30,000-row anonymized sample, pages with `avg_position > 10` and `ctr < 0.02` appear in the declining group at a higher rate than other pages.' — This is a description of a pattern in this specific dataset.

- **Directional:** 'The model score is positively associated with decline risk in this holdout sample.' — Direction is real, magnitude is an estimate, and both may differ on other clients or time periods.

- **Decision-support:** 'Given the current data, pages A, B, and C rank highest for review priority.' — This is an input to a human decision, not the decision itself.

- **Reproducible on this snapshot:** 'All numbers here are computed from `content_refresh_anonymized.csv` and reproducible by running this notebook.' — The claim is tightly scoped to this data.

### What this work CANNOT say

- ❌ **Causal:** 'Updating this page will cause impressions to recover.' — Observational data cannot establish causation. The model identifies correlation only.

- ❌ **Predicting Google's algorithm:** 'Google will rank this page higher if we add 500 words.' — The dataset contains observable outputs (impressions, CTR, position) but not Google's internal ranking signals. No inference about the algorithm is valid.

- ❌ **Generalizing beyond this snapshot:** 'These results hold for all SEO content.' — The data is pseudonymized, from a specific time window (Jan 2025 – Jun 2026), and from a specific client set. Other portfolios may show different patterns.

- ❌ **Guaranteeing ROI:** 'Acting on this queue will increase organic traffic by X%.' — The model scores refresh priority, not expected return. Return depends on editorial quality, competition, and many factors outside this dataset.

---

> **Guiding principle (from the lane guide):** *Tie every claim you make to something the data can actually prove.* If a sentence could appear in a press release, it probably overclaims — rewrite it.

In [4]:
# Sanity check: confirm the data contains only observable signals (no product flags)
print('All columns in the dataset:')
for col in sorted(df.columns):
    print(f'  {col}')

# Confirm no product-decision columns are present (health_score, priority_score, etc.)
forbidden = ['health_score', 'priority_score', 'action_type', 'refresh_tier']
found = [c for c in forbidden if c in df.columns]
print(f'\nProduct-flag columns in dataset: {found if found else "none — clean"}')
print('All features are observable signals: safe to use as model inputs.')

All columns in the dataset:
  age_tier
  age_tier_order
  ai_sessions_90d
  ai_traffic_pct
  avg_position
  char_count
  char_count_tier
  clicks_90d
  clicks_last_30d
  clicks_prev_30d
  client_id
  competition
  competition_level
  content_age_days
  content_id
  content_type
  cpc
  ctr
  days_since_last_update
  days_with_impressions
  days_with_sessions
  engaged_sessions_90d
  engagement_rate
  freshness_tier
  impression_tier
  impressions_90d
  impressions_last_30d
  impressions_prev_30d
  main_intent
  model_used
  pageviews_90d
  position_tier
  provider_used
  scroll_events_90d
  scroll_rate
  search_volume
  sessions_90d
  sessions_last_30d
  sessions_prev_30d
  trend_direction
  trend_pct
  users_90d
  word_count
  word_count_tier

Product-flag columns in dataset: none — clean
All features are observable signals: safe to use as model inputs.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Lane chosen: **Refresh / Content Opportunity Scoring**
- [x] Decision named: *Which pages should an editor review for content refresh first?*
- [x] Action named: *Editor opens top-ranked page and decides whether to rewrite/expand*
- [x] Cost of wrong call named: *False negatives = compounding traffic loss (costlier than false positives)*
- [x] ≥ 2 real numbers shown: baseline P@50, search-volume correlation, position-CTR correlation, word-count medians
- [x] Committed to `work/notebooks/w01_research_question.ipynb` — repo URL submitted on card